# **Liew Shan Yi Part [ HOG + SVM ( RBF Kenel ) ]**

Import Dataset From Kaggle

In [ ]:
import kagglehub
import os
import shutil

target_dir = "dataset"

if not os.path.exists(target_dir):
    print("Dataset folder not found. Downloading...")

    cache_path = kagglehub.dataset_download("divyam6969/chest-xray-pneumonia-dataset")
    print("Downloaded to:", cache_path)

    shutil.copytree(cache_path, target_dir)
    print("Dataset saved to:", target_dir)

else:
    print("Dataset already exists. Skipping download.")

Dataset folder not found. Downloading...


KeyboardInterrupt: 

Check all image / distribution



In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ------------------ set up dir -------------------- #

train_dir = "/content/dataset/train"

test_dir = "/content/dataset/test"

# --------------- process / counting --------------- #

class_counts = {}

for cls in os.listdir(train_dir):
  cls_path = os.path.join(train_dir, cls)

  if os.path.isdir(cls_path):
    class_counts[cls] = len(os.listdir(cls_path))

# ------------- display number of image ------------ #

print("Processed Train Images:")
for cls, count in class_counts.items():
    print(f"  {cls}: {count} images")


test_images = len([img for img in os.listdir(test_dir)
                if img.lower().endswith(('jpg','jpeg','png'))])

print(f"Processed Test Images: {test_images} images")

# --------------------- chart -------------------- #

class_colors = {
    'NORMAL': 'green',
    'BACTERIAL': 'orange',
    'VIRAL': 'red'
}

bar_colors = [class_colors[cls] for cls in class_counts.keys()]

plt.figure(figsize=(6,4))
plt.bar(class_counts.keys(), class_counts.values(), color=bar_colors)
plt.title("Pneumonia Chart")
plt.ylabel("Number of Images")
plt.show()

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True,
)

train_gen = datagen.flow_from_directory(
    train_dir,
    target_size=(224,224),
    class_mode='categorical',
    batch_size=32
)


training + testing with the use of HOG + RBF SVM

In [ ]:
# ==================== Import / from ==================== #

import os
import numpy as np
import cv2
import time
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_validate
from skimage.feature import hog
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import joblib
from joblib import Parallel, delayed
import multiprocessing
from collections import Counter

# ==================== Configure directory ==================== #

train_dir = "/content/dataset/train"
test_dir = "/content/dataset/test"

IMG_SIZE = (96, 96)
HOG_ORIENTATIONS = 9
HOG_PIXELS_PER_CELL = (16, 16)
HOG_CELLS_PER_BLOCK = (2, 2)

# ==================== Extract HOG features ==================== #

def extract_hog_features(image_path, visualize=False):
    """Extract HOG features"""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    img = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_AREA)
    img = cv2.equalizeHist(img)

    if visualize:
        features, hog_image = hog(
            img,
            orientations=HOG_ORIENTATIONS,
            pixels_per_cell=HOG_PIXELS_PER_CELL,
            cells_per_block=HOG_CELLS_PER_BLOCK,
            visualize=True,
            block_norm='L2-Hys'
        )
        return features, hog_image
    else:
        features = hog(
            img,
            orientations=HOG_ORIENTATIONS,
            pixels_per_cell=HOG_PIXELS_PER_CELL,
            cells_per_block=HOG_CELLS_PER_BLOCK,
            visualize=False,
            block_norm='L2-Hys'
        )
        return features

def process_single_image(img_path, label):
    """Process one image"""
    try:
        features = extract_hog_features(img_path)
        if features is not None:
            return features, label
    except:
        pass
    return None, None

def load_dataset_parallel(data_dir, is_test=False):
    """Load images with PARALLEL processing"""
    image_paths = []
    labels = []

    subdirs = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]

    if subdirs:
        print(f"Found class subdirectories: {subdirs}")

        for class_name in subdirs:
            class_path = os.path.join(data_dir, class_name)

            for filename in os.listdir(class_path):
                if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                    img_path = os.path.join(class_path, filename)

                    if 'BACTERIAL' in class_name.upper():
                        label = 'BACTERIAL'
                    elif 'VIRAL' in class_name.upper():
                        label = 'VIRAL'
                    elif 'NORMAL' in class_name.upper():
                        label = 'NORMAL'
                    else:
                        label = class_name.upper()

                    image_paths.append(img_path)
                    labels.append(label)
    else:
        print("No subdirectories found. Parsing filenames...")

        for filename in os.listdir(data_dir):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(data_dir, filename)

                filename_upper = filename.upper()
                if 'NORMAL' in filename_upper:
                    label = 'NORMAL'
                elif 'BACTERIA' in filename_upper:
                    label = 'BACTERIAL'
                elif 'VIRUS' in filename_upper:
                    label = 'VIRAL'
                elif 'PNEUMONIA' in filename_upper:
                    label = 'BACTERIAL'
                else:
                    continue

                image_paths.append(img_path)
                labels.append(label)

    print(f"Processing {len(image_paths)} images in parallel...")
    num_cores = multiprocessing.cpu_count()
    print(f"Using {num_cores} CPU cores")

    results = Parallel(n_jobs=num_cores)(
        delayed(process_single_image)(path, label)
        for path, label in tqdm(zip(image_paths, labels), total=len(image_paths))
    )

# ======================= filtering ======================== #
    X = []
    y = []
    for features, label in results:
        if features is not None:
            X.append(features)
            y.append(label)

    return np.array(X), np.array(y)

# ====================== loading data ====================== #

print("\n" + "="*60)
print("Loading training data (PARALLEL)")
print("="*60)

X_train, y_train = load_dataset_parallel(train_dir)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Feature vector size: {X_train.shape[1]}")

print("\n" + "="*60)
print("Loading test data (PARALLEL)")
print("="*60)

X_test, y_test = load_dataset_parallel(test_dir, is_test=True)

print(f"\nTest set: {X_test.shape[0]} samples")

# ==================== scaling feature ==================== #

print("\nScaling features...")

start_time = time.time()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

end_time = time.time()
elapsed = end_time - start_time

print("Scaling completed!")
print(f"Scaling time: {elapsed:.2f} seconds")

# ==================== CROSS-VALIDATION ==================== #

print("\n" + "="*60)
print("CROSS-VALIDATION EVALUATION")
print("="*60)
print("Running 10-fold stratified cross-validation...")

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

svm_for_cv = SVC(
    C=1.0,
    kernel='rbf',
    gamma='scale',
    class_weight='balanced',
    probability=False,
    cache_size=2000,
    random_state=42
)

print("Computing cross-validation scores...")

cv_scores = cross_val_score(
    svm_for_cv,
    X_train_scaled,
    y_train,
    cv=cv_strategy,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("\n" + "="*60)
print("CROSS-VALIDATION RESULTS")
print("="*60)

# Accuracy results
print(f"\n📊 Accuracy Across 5 Folds:")
for i, score in enumerate(cv_scores, 1):
    print(f"   Fold {i}: {score*100:.2f}%")

print(f"\n{'='*60}")
print(f"Mean CV Accuracy: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")
print(f"{'='*60}")

# ======================= SVM (RBF) ======================= #

print("\n" + "="*60)
print("Training SVM (RBF)")
print("="*60)

svm_classifier = SVC(
    C=1.0,
    kernel='rbf',
    gamma='scale',
    class_weight='balanced',
    probability=False,
    verbose=True,
    cache_size=2000,
    random_state=42
)

start_time = time.time()

print("\nTraining final model...")
svm_classifier.fit(X_train_scaled, y_train)

end_time = time.time()
elapsed = end_time - start_time

print("Training completed!")
print(f"Training time: {elapsed:.2f} seconds")

# ====================== evaluation ====================== #

print("\n" + "="*60)
print("Evaluation")
print("="*60)

y_train_pred = svm_classifier.predict(X_train_scaled)
y_test_pred = svm_classifier.predict(X_test_scaled)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"\nTraining Accuracy: {train_accuracy*100:.2f}%")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Overfitting Gap: {(train_accuracy - test_accuracy)*100:.2f}%")

print("\nTest Classification Report:")
print(classification_report(y_test, y_test_pred))

# =================== Confusion Matrix ================== #
cm_test = confusion_matrix(y_test, y_test_pred)
classes_test = np.unique(y_test)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens',
            xticklabels=classes_test, yticklabels=classes_test)
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# ==================== saving model ==================== #

print("\nSaving model...")
joblib.dump(svm_classifier, "svm_model.joblib", compress=9)
joblib.dump(scaler, 'scaler.joblib')

print("\n" + "="*60)
print("Summary")
print("="*60)
print(f"Training Samples: {len(X_train)}")
print(f"Test Samples: {len(X_test)}")
print(f"Feature Dimensions: {X_train.shape[1]}")
print(f"Training Accuracy: {train_accuracy*100:.2f}%")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

# **Lim Chee Luo Part [ MobileNetV2 ]**

In [ ]:
import kagglehub
import os
import shutil

target_dir = "dataset"

if not os.path.exists(target_dir):
    print("Dataset folder not found. Downloading...")

    cache_path = kagglehub.dataset_download("divyam6969/chest-xray-pneumonia-dataset")
    print("Downloaded to:", cache_path)

    shutil.copytree(cache_path, target_dir)
    print("Dataset saved to:", target_dir)

else:
    print("Dataset already exists. Skipping download.")

Data Exploration & Understanding

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

data_dir = "/content/dataset"
train_dir = f"{data_dir}/train"
test_dir = f"{data_dir}/test"

# ---Count Classes--- #
def count_classes(path):
  classes = os.listdir(path)
  result = {}

  for cls in classes:
    cls_path = os.path.join(path, cls)
    if os.path.isdir(cls_path):
      result[cls] = len(os.listdir(cls_path))

  return result

train_counts = count_classes(train_dir)
train_counts

# --- Plot Class Distribution --- #
plt.figure(figsize=(6,4))
plt.bar(train_counts.keys(), train_counts.values(), color=['blue', 'green', 'orange'])
plt.title("Training Class Distribution")
plt.ylabel("Number of Images")
plt.show()

# --- Image Sizes --- #
sizes = []

for cls in train_counts:
    folder = os.path.join(train_dir, cls)
    for img_name in os.listdir(folder)[:200]:
      img = Image.open(os.path.join(folder, img_name))
      sizes.append(img.size)

# --- Width Distribution --- #
widths = [s[0] for s in sizes]
plt.hist(widths, bins=20)
plt.title("Image Width Distribution")
plt.xlabel("Width")
plt.ylabel("Frequency")
plt.show()

# --- Height Distribution --- #
heights = [s[1] for s in sizes]
plt.hist(heights, bins=20)
plt.title("Image Height Distribution")
plt.xlabel("Height")
plt.ylabel("Frequency")
plt.show()

Test-Folder-Fixing

In [ ]:
import os
import shutil

test_path = "/content/dataset/test"

#create folders for the 3 classes
classes = ["NORMAL", "BACTERIAL", "VIRAL"]
for cls in classes:
    os.makedirs(os.path.join(test_path, cls), exist_ok=True)

#Move files into class folders based on filename
for file in os.listdir(test_path):
    fpath = os.path.join(test_path, file)
    if os.path.isfile(fpath):
      fname = file.lower()
      if "normal" in fname:
        shutil.move(fpath, os.path.join(test_path, "NORMAL", file))
      elif "bacteria" in fname or "bacterial" in fname:
        shutil.move(fpath, os.path.join(test_path, "BACTERIAL", file))
      elif "viral" in fname or "virus" in fname:
        shutil.move(fpath, os.path.join(test_path, "VIRAL", file))

Data preprocessing



In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# --- Data Augmentation for Training --- #
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.15
)

# --- For Test Set (NO augumentation) --- #
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

# --- Data Generators --- #
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset="training"
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset="validation"
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# --- Check Class Indicies --- #
train_generator.class_indices


Model Selection (MobileNetV2)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

#Load MobileNetV2 without the top classification layer
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

#Freeze base model layers
base_model.trainable = False

#Build custom head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(3, activation='softmax')(x)  #3 classes

model = Model(inputs=base_model.input, outputs=output)

#Compile
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

**Model Training**

1- Set Random Seed (Reproducibility)

In [ ]:
import tensorflow as tf
import numpy as np
import random

seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

2- Define Hyperparameters

In [ ]:
EPOCHS = 15
BATCH_SIZE = 32
LR = 0.0001

3- Define Callbacks (Checkpoint, Early Stopping, TensorBoard)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, TensorBoard
import datetime

checkpoint = ModelCheckpoint(
    "best_model.keras",
    save_best_only=True,
    monitor="val_accuracy",
    mode="max",
    verbose=1
)

earlystop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard = TensorBoard(log_dir=log_dir, histogram_freq=1)

4- Train Model

In [ ]:
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=test_generator,
    callbacks=[checkpoint, earlystop, tensorboard]
)

5- Plot Accuracy & Loss Curves

In [ ]:
import matplotlib.pyplot as plt

#Accuracy
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title("Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

#Loss
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

**Model Evaluation**

1- Load Best Model

In [ ]:
from tensorflow.keras.models import load_model

best_model = load_model("best_model.keras")
best_model.save("best_model.keras")

2- Evaluate Model on Test Set

In [ ]:
test_loss, test_acc = best_model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc: .4f}")
print(f"Test Loss: {test_loss: .4f}")

3- Generate Predictions

In [ ]:
import numpy as np

y_pred = best_model.predict(test_generator)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

4- Classification Report

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred_classes, target_names=class_labels))

5- Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels,
            yticklabels=class_labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

6- ROC Curve

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

y_true_bin = label_binarize(y_true, classes=[0,1,2])

plt.figure(figsize=(8,6))

for i in range(3):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{class_labels[i]} (AUC = {roc_auc:.2f})')

plt.plot([0,1], [0,1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title("ROC Curve (Multiclass)")
plt.legend()
plt.show()

7- Error Analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os

# Assume test_generator, y_pred_classes, y_true, class_labels, test_dir are defined from previous cells

# Get the indices of misclassified images
errors = np.where(y_pred_classes != y_true)[0]

plt.figure(figsize=(12,12))

# Get the list of filenames from the generator
# test_generator.filenames contains paths relative to test_dir, e.g., 'NORMAL/image.jpeg'
image_filenames = test_generator.filenames

for i, idx in enumerate(errors[:9]): # Display up to 9 error images
    # Get the relative path for the misclassified image
    relative_path = image_filenames[idx]
    # Construct the full path
    full_path = os.path.join(test_dir, relative_path)

    # Load the image
    img = Image.open(full_path)
    img_array = np.array(img) # Convert PIL Image to NumPy array

    plt.subplot(3,3,i+1)
    plt.imshow(img_array) # Display the original image
    plt.title(f"True: {class_labels[y_true[idx]]}\nPred: {class_labels[y_pred_classes[idx]]}")
    plt.axis('off')

plt.tight_layout() # Adjust layout to prevent overlapping titles
plt.show()

**Model Optimization**

1- Hyperparameter Tuning

In [ ]:
LR_FINE = 1e-5
optimizer = tf.keras.optimizers.Adam(learning_rate=LR_FINE)

2- Fine-Tuning the Base Model

In [ ]:
# Unfreeze last X layers of MobileNetV2
for layer in base_model.layers[-40:]:
    layer.trainable = True

# Compile with lower LR for fine-tuning
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

3- Train Again

In [ ]:
history_finetune = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=5,
    callbacks=[checkpoint, earlystop]
)

4- Add Regularization

In [ ]:
from tensorflow.keras.regularizers import l2

x = Dense(128, activation='relu', kernel_regularizer=l2(0.0001))(x)

5- Final Results

In [ ]:
pred_prob = model.predict(test_generator)
pred = np.argmax(pred_prob, axis=1)
true = test_generator.classes

print(classification_report(true, pred))
cm = confusion_matrix(true, pred)
cm

**Evaluation After Fine-Tuning**

Load Final Tuned Model

In [ ]:
from tensorflow.keras.models import load_model
final_model = load_model("best_model.keras")

Create test generator

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Predict on Test Set

In [ ]:
import numpy as np

y_pred = final_model.predict(test_generator)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = test_generator.classes

class_labels = list(test_generator.class_indices.keys())

Accuracy Curve (After Fine-Tuning)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.plot(history_finetune.history['accuracy'], label='Train Accuracy')
plt.plot(history_finetune.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy After Fine-Tuning')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

Loss Curve (After Fine-Tuning)

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(history_finetune.history['loss'], label='Train Loss')
plt.plot(history_finetune.history['val_loss'], label='Validation Loss')
plt.title('Loss After Fine-Tuning')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
model.save("best_model.keras")

# **Divyadarshini A/P Ramasamy Part [ Custom CNN ]**

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader , random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report , roc_curve, auc
from sklearn.preprocessing import label_binarize
from itertools import cycle
import torch
import torch.nn.functional as F
import kagglehub
import shutil
import math

import torch
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Subset
import collections


**1) DATASET DOWNLOAD & SETUP**

In [ ]:

target_dir = "dataset"

if not os.path.exists(target_dir):
    print("Dataset folder not found. Downloading...")

    cache_path = kagglehub.dataset_download("divyam6969/chest-xray-pneumonia-dataset")
    print("Downloaded to:", cache_path)

    shutil.copytree(cache_path, target_dir)
    print("Dataset saved to:", target_dir)
else:
    print("Dataset already exists. Skipping download.")

test_dir = os.path.join(target_dir, "test")

class_mapping = {
    "normal": "NORMAL",
    "bacteria": "BACTERIAL",
    "virus": "VIRAL"
}

for folder_name in class_mapping.values():
    folder_path = os.path.join(test_dir, folder_name)
    os.makedirs(folder_path, exist_ok=True)


for filename in os.listdir(test_dir):
    file_path = os.path.join(test_dir, filename)
    if os.path.isfile(file_path):
        # Check which keyword is in the filename
        lower_filename = filename.lower()
        for keyword, folder_name in class_mapping.items():
            if keyword in lower_filename:
                dest_path = os.path.join(test_dir, folder_name, filename)
                shutil.move(file_path, dest_path)
                break  # stop checking once matched

print("Test folder organized into class subfolders: NORMAL, BACTERIA, VIRAL.")


**2) TRANSFORMS & DATASET**

Helper function for plotting distributions

In [ ]:
def plot_distribution(class_counts, class_names, title):
    """
    Plots a bar chart of the class distribution.

    Args:
        class_counts (collections.Counter): A Counter object mapping class index to count.
        class_names (list): A list of the actual class names.
        title (str): The title for the plot.
    """

    counts = [class_counts[i] for i in sorted(class_counts.keys())]
    names = [class_names[i] for i in sorted(class_counts.keys())]
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=(10, 6))

    bars = ax.bar(names, counts, color=plt.cm.Paired(np.arange(len(names))))
    ax.set_title(title, fontsize=16)
    ax.set_ylabel('Number of Samples', fontsize=12)
    ax.set_xlabel('Class', fontsize=12)
    plt.xticks(rotation=45, ha="right")

    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2.0, yval + 5, int(yval),
                ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.show()

Define Transformers

In [ ]:
class TransformedSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

# --- Define Transforms ---
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Dataset Loading and Splitting

In [ ]:
train_dir = "dataset/train"
full_train_dataset = datasets.ImageFolder(train_dir, transform=None)

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
torch.manual_seed(42)
train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size])

train_dataset = TransformedSubset(train_subset, transform=train_transforms)
val_dataset = TransformedSubset(val_subset, transform=val_transforms)

print("\n--- Dataset Summary ---")
print("Total images:", len(full_train_dataset))
print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Training Set Class Exploration

In [ ]:
print("\n--- Class Distribution in Training Set (Before Balancing) ---")
train_targets = [label for _, label in train_subset]
class_counts_before = collections.Counter(train_targets)
class_names = full_train_dataset.classes

for i in sorted(class_counts_before.keys()):
    print(f"Class '{class_names[i]}': {class_counts_before[i]} samples")

# --- VISUALIZATION (BEFORE) ---
plot_distribution(
    class_counts=class_counts_before,
    class_names=class_names,
    title="Class Distribution in Training Set (Before Balancing)"
)

Addressing data imbalance with WeightedRandomSampler

In [ ]:
class_counts = np.bincount(train_targets)
class_weights = 1. / class_counts
sample_weights = np.array([class_weights[t] for t in train_targets])
sampler = torch.utils.data.WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).double(),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("\nDataLoaders created.")




Class Distribution in one epoch from DataLoader (After Balancing)

In [ ]:
train_targets_after = []
for _, labels in train_loader:
    train_targets_after.extend(labels.numpy())

class_counts_after = collections.Counter(train_targets_after)

for i in sorted(class_counts_after.keys()):
    print(f"Class '{class_names[i]}': {class_counts_after[i]} samples drawn")

# --- VISUALIZATION (AFTER) ---
plot_distribution(
    class_counts=class_counts_after,
    class_names=class_names,
    title="Class Distribution Drawn in One Epoch (With WeightedRandomSampler)"
)

**3) MODEL DEFINITION**

In [ ]:

class ChestXrayCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(ChestXrayCNN, self).__init__()

        def conv_block(in_channels, out_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(),
                nn.MaxPool2d(2,2)
            )

        self.block1 = conv_block(3, 32)
        self.block2 = conv_block(32, 64)
        self.block3 = conv_block(64, 128)
        self.block4 = conv_block(128, 256)

        self.gap = nn.AdaptiveAvgPool2d((1,1))

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)
        x = self.fc(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = ChestXrayCNN(num_classes=3).to(device)

 **4) PYTORCH LR FINDER**

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def lr_find(model, loader, optimizer, criterion, start_lr=1e-7, end_lr=10):
    model.train()
    num_batches = len(loader)
    lr_mult = (end_lr/start_lr)**(1/num_batches)
    lr = start_lr
    optimizer.param_groups[0]["lr"] = lr

    losses = []
    lrs = []

    for batch_i, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        lrs.append(lr)

        lr *= lr_mult
        optimizer.param_groups[0]["lr"] = lr

        if batch_i % 20 == 0:
            print(f"Batch {batch_i}/{num_batches} LR={lr:.6f} Loss={loss.item():.4f}")

    plt.plot(lrs, losses)
    plt.xscale("log")
    plt.xlabel("Learning Rate")
    plt.ylabel("Loss")
    plt.title("LR Finder")
    plt.show()


print("\nRunning LR Finder...")
lr_find(model, train_loader, optimizer, criterion)

**5) LOSS, OPTIMIZER, ONE CYCLE LR**

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 20
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=num_epochs
)

**6) Model Training**

In [ ]:
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []
best_val_accuracy = 0

for epoch in range(num_epochs):

    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    avg_train_loss = train_loss / len(train_loader)
    train_acc = 100 * train_correct / train_total

    train_losses.append(avg_train_loss)
    train_accuracies.append(train_acc)


    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    avg_val_loss = val_loss / len(val_loader)
    val_acc = 100 * val_correct / val_total

    val_losses.append(avg_val_loss)
    val_accuracies.append(val_acc)

    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val   Loss: {avg_val_loss:.4f} | Val   Acc: {val_acc:.2f}%")
    print("-"*50)

    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        torch.save(model.state_dict(), "best_chest_xray_model.pth")
        print(f"Best Model Saved! New Best Val Acc = {val_acc:.2f}%")

print("\nTraining Complete!")
print(f"Best Validation Accuracy: {best_val_accuracy:.2f}%")

**7) CONFUSION MATRIX + ERROR ANALYSIS**

In [ ]:
class Interpretation:
    def __init__(self, model, loader,class_names, criterion):
        self.model = model
        self.loader = loader
        self.criterion = criterion
        self.class_names = class_names
        self.preds = []
        self.targets = []
        self.losses = []
        self.probs = []


    def evaluate(self):
        self.model.eval()
        self.preds.clear()
        self.targets.clear()
        self.losses.clear()

        with torch.no_grad():
            for images, labels in self.loader:
                images, labels = images.to(device), labels.to(device)
                outputs = self.model(images)

                probabilities = F.softmax(outputs, dim=1)
                self.probs.extend(probabilities.cpu().numpy())
                loss = F.cross_entropy(outputs, labels, reduction='none')
                _, predicted = torch.max(outputs, 1)

                self.preds.extend(predicted.cpu().numpy())
                self.targets.extend(labels.cpu().numpy())
                self.losses.extend(loss.cpu().numpy())

    def plot_confusion_matrix(self):
        cm = confusion_matrix(self.targets, self.preds)
        disp = ConfusionMatrixDisplay(cm, display_labels=self.class_names)
        disp.plot(cmap='Blues', xticks_rotation=45)
        plt.title("Confusion Matrix")
        plt.show()

    def plot_top_losses(self, k=9):
        losses = np.array(self.losses)
        preds = np.array(self.preds)
        targets = np.array(self.targets)

        top_loss_indices = np.argsort(losses)[-k:]

        fig, axes = plt.subplots(int(math.ceil(k/3)), 3, figsize=(15, 10))
        axes = axes.flatten()

        for i, idx in enumerate(top_loss_indices):
            image, _ = self.loader.dataset[idx]

            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            image = image.permute(1, 2, 0).numpy() * std + mean
            image = np.clip(image, 0, 1)

            ax = axes[i]
            ax.imshow(image)
            pred_class = self.class_names[preds[idx]]
            actual_class = self.class_names[targets[idx]]
            ax.set_title(f"Pred: {pred_class}\nActual: {actual_class}\nLoss: {losses[idx]:.2f}")
            ax.axis('off')

        plt.suptitle("Top Losses", fontsize=16)
        plt.tight_layout()
        plt.show()

    def plot_roc_curve(self):
        y_true = np.array(self.targets)
        y_prob = np.array(self.probs)
        n_classes = len(self.class_names)


        y_true_binarized = label_binarize(y_true, classes=range(n_classes))

        fpr = dict()
        tpr = dict()
        roc_auc = dict()

        for i in range(n_classes):
            fpr[i], tpr[i], _ = roc_curve(y_true_binarized[:, i], y_prob[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])

        plt.figure(figsize=(8, 6))
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

        for i, color in zip(range(n_classes), colors):
            plt.plot(fpr[i], tpr[i], color=color, lw=2,
                    label=f'{self.class_names[i]} (AUC = {roc_auc[i]:.2f})')

        plt.plot([0, 1], [0, 1], 'k--', lw=2)

        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC Curve (Multiclass - One-vs-Rest)')
        plt.legend(loc="lower right")
        plt.grid(False)
        plt.show()


7.1)  Test Dataset

In [ ]:
test_dir = "dataset/test"

test_dataset = datasets.ImageFolder(test_dir, transform=val_transforms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

interp = Interpretation(model, test_loader, test_loader.dataset.classes, criterion)

print("Running evaluation on the test set...")
interp.evaluate()

7.1.1) Confusion Matrix

In [ ]:
print("Confusion Matrix of Test Data")
interp.plot_confusion_matrix()
print("\n Classification Report:\n")
print(classification_report(interp.targets, interp.preds, target_names=interp.class_names))

7.1.2) Qualitative Analysis of Misclassified Images of Test Data

In [ ]:
print("Qualitative Analysis of Misclassified Images of Test Data")
interp.plot_top_losses(k=6)

7.1.3) ROC Curve for Test Data

In [ ]:
print("ROC Curve for Test Data")
interp.plot_roc_curve()


7.2) VALIDATION DATASET

In [ ]:
interp = Interpretation(model, val_loader,full_train_dataset.classes, criterion)

print("Running evaluation on the Vaidation set...")
interp.evaluate()

7.2.1) Confusion Matrix

In [ ]:
print("Confusion Matrix of Validation Data")
interp.plot_confusion_matrix()
print("\n Validation Set Classification Report:\n")
print(classification_report(interp.targets, interp.preds, target_names=full_train_dataset.classes))

7.2.2) Qualitative Analysis of Misclassified Images of Test Data

In [ ]:
print("Qualitative Analysis of Misclassified Images of Validation Data")
interp.plot_top_losses(k=6)

7.2.3) ROC Curve for Validation Data

In [ ]:
print("ROC Curve for Validation Data")
interp.plot_roc_curve()


**8) LEARNING CURVES**

8.1) Loss Curve

In [ ]:
plt.figure(figsize=(30,8))
plt.subplot(1,2,1)
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()
plt.title("Loss Curve")

8.2) Accuracy Curve

In [ ]:
plt.figure(figsize=(30,8))
plt.subplot(1,2,2)
plt.plot(train_accuracies, label="Train Acc")
plt.plot(val_accuracies, label="Val Acc")
plt.legend()
plt.title("Accuracy Curve")

**9) Import the Model and test with an image**

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import os

CLASS_NAMES = ['BACTERIAL', 'NORMAL', 'VIRAL']


# MODEL ARCHITECTURE
class ChestXrayCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(ChestXrayCNN, self).__init__()

        def conv_block(in_channels, out_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(),
                nn.MaxPool2d(2,2)
            )

        self.block1 = conv_block(3, 32)
        self.block2 = conv_block(32, 64)
        self.block3 = conv_block(64, 128)
        self.block4 = conv_block(128, 256)

        self.gap = nn.AdaptiveAvgPool2d((1,1))

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)
        x = self.fc(x)
        return x



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ChestXrayCNN(num_classes=3)

model_path = "best_chest_xray_model.pth"

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Weights loaded from {model_path}")
else:
    print(f"Error: Could not find {model_path}")
    exit()

model.to(device)
model.eval()



# PREPROCESSING
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])



# PREDICTION FUNCTION
def predict_image(image_path):
    try:
        image = Image.open(image_path).convert('RGB')
    except Exception as e:
        print(f"Could not open image {image_path}: {e}")
        return

     # Show the image
    plt.figure(figsize=(5,5))
    plt.imshow(image)
    plt.axis('off')


    # Transform Image
    input_tensor = inference_transform(image)

    # 3. Add Batch Dimension
    input_batch = input_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_batch)
        probabilities = torch.nn.functional.softmax(output[0], dim=0)
        _, predicted_idx = torch.max(output, 1)

        confidence = probabilities[predicted_idx].item()
        label = CLASS_NAMES[predicted_idx.item()]


    print(f"\n--- Result for: {os.path.basename(image_path)} ---")
    print(f"Prediction: {label.upper()}")
    print(f"Confidence: {confidence * 100:.2f}%")

    print("\nDetailed Probabilities:")
    for i, class_name in enumerate(CLASS_NAMES):
        print(f"  {class_name}: {probabilities[i].item() * 100:.2f}%")



# 6. Execution
# Image Path , Example: "C:/Users/Downloads/test_xray.jpeg"
test_image_path = "testing_image.jpeg"

predict_image(test_image_path)